# DenseNet121 Transfer Learning — Head Only, Filtered Images (3k/class)

DenseNet121 head-only fine-tuning on the filtered NASA GLOBE dataset.
Backbone is fully frozen; only the final classifier layer is trained.
Images are selected from the ResNet18 cloud filter CSV using per-class `head2_conf` thresholds,
randomly sampled across the full confidence range (random_state=42).

Differences from `12_transfer_learning_densenet121.ipynb`:
- Max 3,000 images per class (down from 7,000)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

CSV_PATH    = Path("../resources/cloud-images/NASA_GLOBE_CD/cloud_filter_results_1.csv")
IMAGES_ROOT = Path("../resources/cloud-images/NASA_GLOBE_CD/downloaded_images")

THRESHOLDS = {
    "Ac": 0.8,
    "As": 0.8,
    "Cb": 0.8,
    "Cc": 0.8,
    "Ci": 0.8,
    "Cs": 0.8,
    "Cu": 0.8,
    "Ns": 0.8,
    "Sc": 0.8,
    "St": 0.8,
}

def index_labeled_images_filtered(csv_path, thresholds, images_root, max_per_class=3000):
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"cloud_conf": "head2_conf", "head1_class": "head1_pred"})
    labeled_images = {}
    for folder, thresh in thresholds.items():
        clean = df[(df["folder"] == folder) & (df["head2_conf"] >= thresh)]
        clean = clean.sample(min(max_per_class, len(clean)), random_state=42)
        for _, row in clean.iterrows():
            labeled_images[row["filename"]] = {
                "label": folder,
                "path":  str(Path(images_root) / folder / row["filename"]),
            }
    return labeled_images

labeled_images = index_labeled_images_filtered(CSV_PATH, THRESHOLDS, IMAGES_ROOT)

counts = Counter(v["label"] for v in labeled_images.values())
for folder in sorted(counts):
    print(f"  {folder:6s}  {counts[folder]:,}")
print(f"  {'TOTAL':6s}  {sum(counts.values()):,}")

In [2]:
def extract_labels(labeled_images):
    paths, labels = [], []
    for image_name in labeled_images:
        paths.append(labeled_images[image_name]["path"])
        labels.append(labeled_images[image_name]["label"])
    return paths, np.array(labels)

paths, labels = extract_labels(labeled_images)

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"device: {device}")

device: mps


In [4]:
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMG_SIZE = (224, 224)
MAX_FRAC = 0.14

border_translation = T.RandomAffine(
    degrees=0,
    translate=(MAX_FRAC, MAX_FRAC),
    interpolation=InterpolationMode.BILINEAR,
    fill=0
)

# T.Lambda is fine locally with num_workers=0
wrap_translation = T.Lambda(lambda x: torch.roll(
    x,
    shifts=(
        int(torch.randint(-int(MAX_FRAC * x.shape[-2]), int(MAX_FRAC * x.shape[-2]) + 1, (1,)).item()),
        int(torch.randint(-int(MAX_FRAC * x.shape[-1]), int(MAX_FRAC * x.shape[-1]) + 1, (1,)).item()),
    ),
    dims=(-2, -1),
))

stacked = T.Compose([
    T.RandomChoice([wrap_translation, border_translation]),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomAffine(
        degrees=20, scale=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR, fill=0
    ),
    T.RandomResizedCrop(
        size=IMG_SIZE, scale=(0.80, 1.00), ratio=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR
    ),
])

one_of = T.RandomChoice([
    wrap_translation,
    border_translation,
    T.RandomChoice([T.RandomHorizontalFlip(p=1.0), T.RandomVerticalFlip(p=1.0)]),
    T.RandomRotation(degrees=20, interpolation=InterpolationMode.BILINEAR, fill=0),
])

normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.RandomChoice([stacked, one_of]),
    normalize,
])

eval_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    normalize,
])

In [5]:
import torchvision
import torch.nn as nn

weights = torchvision.models.DenseNet121_Weights.IMAGENET1K_V1
model = torchvision.models.densenet121(weights=weights).to(device)
print("DenseNet121 loaded")

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /Users/lucasnseyep/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:05<00:00, 5.96MB/s]


DenseNet121 loaded


In [6]:
from sklearn.preprocessing import LabelEncoder

def encode_labels(labels):
    ordinal_encoder = LabelEncoder()
    encoded_labels = ordinal_encoder.fit_transform(labels)
    return encoded_labels, ordinal_encoder.classes_

encoded_labels, class_names = encode_labels(labels)
print(f"Classes ({len(class_names)}): {class_names}")

Classes (10): ['Ac' 'As' 'Cb' 'Cc' 'Ci' 'Cs' 'Cu' 'Ns' 'Sc' 'St']


In [7]:
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from PIL import Image

class MyImages(Dataset):
    def __init__(self, paths, encoded_labels, split="train", test_size=0.2, val_size=0.1,
                 random_state=42, transform=None, split_indices=None):
        if split not in {None, "train", "val", "test"}:
            raise ValueError(f"split must be one of {{'None','train','val','test'}}, got {split!r}")

        self.transform = transform
        paths = np.array(list(paths))
        encoded_labels = np.array(list(encoded_labels))

        if split_indices is None:
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
            trainval_idx, test_idx = next(sss1.split(paths, encoded_labels))

            val_within_trainval = val_size / (1.0 - test_size)
            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_within_trainval, random_state=random_state)
            train_rel_idx, val_rel_idx = next(sss2.split(paths[trainval_idx], encoded_labels[trainval_idx]))

            split_indices = {
                "train": trainval_idx[train_rel_idx],
                "val":   trainval_idx[val_rel_idx],
                "test":  test_idx,
            }

        idx = split_indices[split]
        self.paths = paths[idx].tolist()
        self.encoded_labels = encoded_labels[idx].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.encoded_labels[idx]


paths_arr  = np.array(list(paths))
labels_arr = np.array(list(encoded_labels))

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(sss1.split(paths_arr, labels_arr))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1/0.8, random_state=42)
train_rel_idx, val_rel_idx = next(sss2.split(paths_arr[trainval_idx], labels_arr[trainval_idx]))

split_indices = {
    "train": trainval_idx[train_rel_idx],
    "val":   trainval_idx[val_rel_idx],
    "test":  test_idx,
}

train_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="train", transform=train_transforms,
                     split_indices=split_indices)
valid_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="val",   transform=eval_transforms,
                     split_indices=split_indices)
test_set  = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="test",  transform=eval_transforms,
                     split_indices=split_indices)

print(f"train: {len(train_set):,}  val: {len(valid_set):,}  test: {len(test_set):,}")

train: 49,000  val: 7,000  test: 14,000


In [8]:
from torch.utils.data import DataLoader, WeightedRandomSampler

train_labels_list = [train_set.encoded_labels[i] for i in range(len(train_set))]
class_counts  = np.bincount(train_labels_list)
class_weights = 1.0 / class_counts
sample_weights = torch.tensor([class_weights[l] for l in train_labels_list], dtype=torch.float)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# num_workers=0 required locally — T.Lambda in transforms cannot be pickled by worker processes
train_loader = DataLoader(train_set, batch_size=64, sampler=sampler, num_workers=0)
valid_loader = DataLoader(valid_set, batch_size=64, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=64, num_workers=0)

In [9]:
n_classes = len(class_names)  # 10

# Freeze entire backbone
for param in model.parameters():
    param.requires_grad = False

# Replace classifier with a new head — only this will be trained
# DenseNet121 backbone outputs 1024 features into model.classifier
model.classifier = nn.Linear(model.classifier.in_features, n_classes).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

Trainable params: 10,250 / 6,964,106


In [10]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=20, checkpoint_path="best_model.pt"):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    best_val = 0.0
    epochs_without_improvement = 0

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        # Keep model in eval mode to fix BatchNorm running stats in the frozen backbone.
        # The classifier head is a plain Linear layer so train vs eval makes no difference for it.
        model.eval()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)

        train_loss = total_loss / len(train_loader)
        train_acc  = metric.compute().item()
        val_acc    = evaluate_tm(model, valid_loader, metric).item()

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_acc)
        history["valid_metrics"].append(val_acc)

        print(f"Epoch {epoch + 1}/{n_epochs} | "
              f"loss: {train_loss:.4f} | "
              f"train: {train_acc:.4f} | "
              f"val: {val_acc:.4f}"
              + (" *" if val_acc > best_val else ""))

        if val_acc > best_val:
            best_val = val_acc
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch + 1} (best val: {best_val:.4f})")
                break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return history

In [ ]:
n_epochs  = 100
optimizer = torch.optim.AdamW(model.classifier.parameters())
xentropy  = nn.CrossEntropyLoss()
accuracy  = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes).to(device)

history = train(
    model, optimizer, xentropy, accuracy,
    train_loader, valid_loader,
    n_epochs, patience=20,
    checkpoint_path="best_model_12_1_densenet121.pt",
)

test_acc = evaluate_tm(model, test_loader, accuracy)
print(f"\nTest accuracy: {test_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history["train_losses"], label="train loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training loss")
axes[0].legend()

axes[1].plot(history["train_metrics"], label="train acc")
axes[1].plot(history["valid_metrics"], label="val acc")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()